In [ ]:
#| default_exp skill
#| export
from nbskill.edit import edit_notebook
from nbskill.execute import exec_nb
from nbskill.foundation import generated_owner
from nbskill.knowledge import reference_query
from nbskill.read import context
from nbskill.review import diff_nb, style_check

In [ ]:
from pathlib import Path
from nbskill import skill
from tempfile import gettempdir
from fastcore.nbio import mk_cell, read_nb
from nbskill.foundation import write_demo_notebook
from nbskill.skill import context

# Native nbskill Pyskill

> Direct notebook operations for coding agents.

#| export
`nbskill.skill` is the short, direct API for notebook-owned code. Import it with `from nbskill.skill import *`. It is the source of truth for the notebook workflow; MCP remains available for clients that need a tool adapter.

#| export
## Work with a notebook

Call `generated_owner(path)` before touching a Python file. A returned notebook means the Python file is generated and must not be edited. Call `context` to read the relevant notebook cells. For a change that needs prior art, call `reference_query` before choosing an implementation. Use `edit_notebook` for the one structured mutation.

Prove the result in the same change loop. `exec_nb` runs the affected notebook scope without writing outputs when `check_only=True`. `diff_nb` shows the code-cell change, and `style_check` reports source diagnostics. Ordinary Python files remain on the normal coding path.

In [ ]:
#| exportd
summary = context("nbs/01_read.ipynb#context", scope="nbs", view="summary", verbose=False)
assert summary["kind"] == "context"

### Direct workflow check

This test uses the public Pyskill to make a structured change to a fresh notebook. It protects the native path from quietly depending on MCP.

In [ ]:
#| hide
with write_demo_notebook(
    "14_pyskill_workflow.ipynb",
    base=gettempdir(),
    cells=[
        mk_cell("## Answer", cell_type="markdown"),
        mk_cell("answer = 41", cell_type="code"),
        mk_cell("assert answer == 42", cell_type="code"),
    ],
) as path:
    answer = read_nb(path).cells[1]
    assert skill.edit_notebook(
        str(path),
        [dict(op="replace_text", cell_id=answer.id, old="answer = 41", new="answer = 42")],
        auto_feedback=False,
    )["changed"]
    assertion = read_nb(path).cells[2]
    skill.exec_nb(str(path), up2id=assertion.id, check_only=True, allow_new=True, show_output=False)
    assert "+answer = 42" in skill.diff_nb(str(path), ref_a=None, ref_b=None)

In [ ]:
#| hide
expected = {
    "context", "generated_owner", "reference_query", "edit_notebook",
    "exec_nb", "diff_nb", "style_check",
}
assert set(skill.__all__) == expected
assert "Call `context` to read the relevant notebook cells." in skill.__doc__
assert skill.__doc__.count("summary = context(") == 1
assert skill.generated_owner(Path(skill.__file__)).name == "14_pyskill.ipynb"

handwritten = Path(gettempdir()) / "14_pyskill_handwritten.py"
handwritten.write_text("value = 1\n")
try: assert skill.generated_owner(handwritten) is None
finally: handwritten.unlink()